In [1]:
import numpy as np
import torch
import os
import scipy.io as sio
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader,random_split
from sklearn import preprocessing
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
import torch.nn as nn
from torch.nn import functional as F
import gc
from optparse import OptionParser
from tqdm import tqdm
import time
import csv
import pandas as pd
# from main_train import *
# from main_test import *
from Data_loader import *
#from architectures.Network import *
from metrics_util import * 
#from architectures.FNO3dTo2d import FNO3dTo2d
#from architectures.LTAE_Net import MRE_UTAE
#from architectures.UpdatedNetwork import Net
#from architectures.SimpleTFusionUNet3D import SimpleTFusionUNet3D
#from architectures.UNO import UNOWithTemporalReduction
#from architectures.arch_utils import *
from train_functions import *
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
import os

base_dir = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet"
target_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if "weights.pth" in file  and ("bs64" in root or "bs64" in file):
            target_files.append(os.path.join(root, file))

print(f"Found {len(target_files)} matching weights.pth files:")
for f in target_files:
    print(f)


Found 4 matching weights.pth files:
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/hom/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/hom/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth


In [7]:
path_A = "/storage/coda1/p-jueda3/0/hnieves6/MRE_Inversion/Model_Logs/FDTDNet/mse/bs64weights.pth"


ckpt_A = torch.load(path_A, map_location=device)
model_A = setup_model("FDTDNet")[0]
model_A.load_state_dict(ckpt_A["state_dict"])
model_A.eval()

path_B = "/storage/coda1/p-jueda3/0/hnieves6/MRE_Inversion/Model_Logs/FDTDNet/hom/lam1_0/bs64weights.pth"


ckpt_B = torch.load(path_B, map_location=device)
model_B = setup_model("FDTDNet")[0]
model_B.load_state_dict(ckpt_B["state_dict"])
model_B.eval()

NameError: name 'device' is not defined

In [8]:
root = os.path.dirname(os.getcwd())
train_input = '/Training'
val_input = '/Validation'
test_input = '/Test'
model = '/Model_Logs'
# train_input= root + train_input + '/'
# val_input= root + val_input + '/'
# dir_model= root + model + '/'
batch_size=18
epochs=3
lr=0.001
offsets=8
fov=0.2


sample_path = root+'/MRE_PInversion/SingleSampleForCodeDebugging/'


debug_loader = get_dataloader_for_train(sample_path, offsets, fov, batch_size=3)
# Get sample to test
for batch in debug_loader:
    wave_tensors_debug, mu, gt_debug, mfre_debug, fov, index = batch
    break
wave_tensors_debug = wave_tensors_debug.to(device)
gt_debug = gt_debug.to(device)
mfre_debug = mfre_debug.to(device)
fov = fov.to(device)

model_inputs = wave_tensors_debug.permute(4, 0, 1, 2, 3)
model_inputs = model_inputs.permute(1,2,0,3,4)

FileNotFoundError: [Errno 2] No such file or directory: '/storage/project/r-jueda3-0/jueda3/MRE_OAGH_test/MinimalCode_3_5_2026/MRE_PInversion/SingleSampleForCodeDebugging/'

In [ ]:
pred_A = model_A(model_inputs)
pred_B = model_B(model_inputs)

In [ ]:
err_A = (pred_A - gt_debug).abs()
err_B = (pred_B - gt_debug).abs()
diff_AB = pred_A - pred_B

In [ ]:
print(predicted_stiffness_A.shape)

In [ ]:
with torch.no_grad():
    # ---- Convert wave number → stiffness ----
    stiffness_A = wave_number_to_shear_stiffness(pred_A, mfre_debug, fov)
    stiffness_B = wave_number_to_shear_stiffness(pred_B, mfre_debug, fov)
    stiffness_gt = wave_number_to_shear_stiffness(gt_debug, mfre_debug, fov)


In [ ]:
def visualize_batch(
    preds_A,
    preds_B=None,
    gt=None,
    batch_indices=None,
    title_prefix="Sample",
    vmin=None,
    vmax=None
):
    """
    preds_A, preds_B, gt: torch.Tensor [B, 1, H, W]
    batch_indices: list of batch indices to plot (default: all)
    """

    preds_A = preds_A.detach().cpu()
    preds_B = preds_B.detach().cpu() if preds_B is not None else None
    gt = gt.detach().cpu() if gt is not None else None

    B = preds_A.shape[0]
    if batch_indices is None:
        batch_indices = range(B)

    for b in batch_indices:
        rows = 1 + (preds_B is not None) + (gt is not None)
        cols = 3 if preds_B is not None else 1
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        axes = np.atleast_2d(axes)

        r = 0

        # ---- Ground Truth ----
        if gt is not None:
            im = axes[r, 0].imshow(gt[b, 0], cmap="viridis", vmin=vmin, vmax=vmax)
            axes[r, 0].set_title(f"{title_prefix} {b} – GT")
            axes[r, 0].axis("off")
            plt.colorbar(im, ax=axes[r, 0])
            r += 1

        # ---- Model A ----
        im = axes[r, 0].imshow(preds_A[b, 0], cmap="viridis", vmin=vmin, vmax=vmax)
        axes[r, 0].set_title(f"{title_prefix} {b} – Model A")
        axes[r, 0].axis("off")
        plt.colorbar(im, ax=axes[r, 0])

        if preds_B is not None:
            im = axes[r, 1].imshow(preds_B[b, 0], cmap="viridis", vmin=vmin, vmax=vmax)
            axes[r, 1].set_title("Model B")
            axes[r, 1].axis("off")
            plt.colorbar(im, ax=axes[r, 1])

            diff = preds_A[b, 0] - preds_B[b, 0]
            im = axes[r, 2].imshow(diff, cmap="coolwarm")
            axes[r, 2].set_title("A − B")
            axes[r, 2].axis("off")
            plt.colorbar(im, ax=axes[r, 2])

        plt.tight_layout()
        plt.show()


In [ ]:
visualize_batch(
            preds_A=pred_A,
            preds_B=pred_B,
            gt=gt_debug,
            batch_indices=[0, 1, 2],  # show first 3 samples
            vmin=0,
            vmax=gt_debug.max().item()
        )

In [ ]:
def visualize_stiffness_batch(
    stiffness_A,
    stiffness_B=None,
    stiffness_gt=None,
    batch_indices=None,
    title_prefix="Sample",
    vmin=None,
    vmax=None,
):
    """
    stiffness_*: torch.Tensor [B, 1, H, W]
    """

    stiffness_A = stiffness_A.detach().cpu()
    stiffness_B = stiffness_B.detach().cpu() if stiffness_B is not None else None
    stiffness_gt = stiffness_gt.detach().cpu() if stiffness_gt is not None else None

    B = stiffness_A.shape[0]
    if batch_indices is None:
        batch_indices = range(B)

    for b in batch_indices:
        rows = 1 + (stiffness_gt is not None)
        cols = 3 if stiffness_B is not None else 1

        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        axes = np.atleast_2d(axes)

        r = 0

        # ---- Ground Truth ----
        if stiffness_gt is not None:
            im = axes[r, 0].imshow(
                stiffness_gt[b, 0],
                cmap="viridis",
                vmin=vmin,
                vmax=vmax
            )
            axes[r, 0].set_title(f"{title_prefix} {b} – GT μ")
            axes[r, 0].axis("off")
            plt.colorbar(im, ax=axes[r, 0])
            r += 1

        # ---- Model Predictions ----
        im = axes[r, 0].imshow(
            stiffness_A[b, 0],
            cmap="viridis",
            vmin=vmin,
            vmax=vmax
        )
        axes[r, 0].set_title("Model A μ")
        axes[r, 0].axis("off")
        plt.colorbar(im, ax=axes[r, 0])

        if stiffness_B is not None:
            im = axes[r, 1].imshow(
                stiffness_B[b, 0],
                cmap="viridis",
                vmin=vmin,
                vmax=vmax
            )
            axes[r, 1].set_title("Model B μ")
            axes[r, 1].axis("off")
            plt.colorbar(im, ax=axes[r, 1])

            diff = stiffness_A[b, 0] - stiffness_B[b, 0]
            im = axes[r, 2].imshow(diff, cmap="coolwarm")
            axes[r, 2].set_title("μ(A − B)")
            axes[r, 2].axis("off")
            plt.colorbar(im, ax=axes[r, 2])

        plt.tight_layout()
        plt.show()


In [ ]:
visualize_stiffness_batch(
            stiffness_A,
            stiffness_B,
            stiffness_gt,
            batch_indices=[0, 1, 2],
            vmin=0,
            vmax=stiffness_gt.max().item()
        )

In [ ]:
# evaluator.py
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from Data_loader import *
from evaluation.metrics import compute_mae, compute_rmse, compute_ssim, compute_cnr, get_masks


def capture_fixed_samples(model, test_loader, device, num_samples=10):
    """
    Capture a fixed number of samples for inspection across models/SNRs.
    Guarantees same ordering.
    """
    model.eval()
    saved = []

    with torch.no_grad():
        for i, (wave, gt, mfre, fov, index) in enumerate(test_loader):
            if i >= num_samples:
                break

            wave = wave.to(device)
            gt   = gt.to(device)

            pred = model(wave)

            saved.append({
                "wave": wave.cpu(),
                "gt": gt.cpu(),
                "pred": pred.cpu(),
                "index": index,
                "mfre": mfre,
                "fov": fov
            })

    return saved


def evaluate_model_metrics(model, test_loader, device, has_inclusions=False):
    """
    Evaluate a model over the entire DataLoader and compute metrics using functions in metrics.py.
    Returns a dict of averaged metrics.
    """
    model.eval()
    out = {
        "mae_k": [],
        "rmse_k": [],
        "ssim_k": [],
        "mae_mu": [],
        "rmse_mu": [],
        "ssim_mu": [],
        "cnr": [],
        
    }

    with torch.no_grad():
        for wave, mu, gt, mfre, fov, index in tqdm(test_loader, desc="Evaluating"):
            wave, mu, gt, mfre, fov = wave.to(device), mu.to(device), gt.to(device), mfre.to(device, dtype=wave.dtype), fov.to(device, dtype=wave.dtype)
            wave = wave.to(device)
            gt   = gt.to(device)
            wave = wave.permute(4, 0, 1, 2, 3)
            wave = wave.permute(1,2,0,3,4) 
            pred_k = model(wave)
            predicted_stiffness = wave_number_to_shear_stiffness(pred_k, mfre, fov)
            gt_stiffness = mu
            
            # Compute metrics
            out["mae_k"].append(compute_mae(pred_k, gt))
            out["rmse_k"].append(compute_rmse(pred_k, gt))
            out["ssim_k"].append(compute_ssim(pred_k, gt))
            out["mae_mu"].append(compute_mae(predicted_stiffness, gt_stiffness))
            out["rmse_mu"].append(compute_rmse(predicted_stiffness, gt_stiffness))
            out["ssim_mu"].append(compute_ssim(predicted_stiffness, gt_stiffness))
            
            if has_inclusions:
                inc_mask, bg_mask = get_masks(gt_stiffness)
                out["cnr"].append(compute_cnr(predicted_stiffness.squeeze(), inc_mask, bg_mask))

    # Average across batches
    return {k: np.mean(v) for k, v in out.items() if len(v) > 0}


def run_full_evaluation(models, snr_levels, test_dir, device, batch_size=1, has_inclusions=False):
    """
    Sweep over multiple models and SNR levels.
    Returns a tidy pandas DataFrame.
    """
    rows = []

    for model_name, model in models.items():
        for snr in snr_levels:
            loader = get_dataloader_for_test(
                test_dir,
                batch_size=batch_size,
                snr_db=snr
            )

            metrics = evaluate_model_metrics(
                model,
                loader,
                device,
                has_inclusions=has_inclusions
            )

            row = {
                "model": model_name,
                "snr_db": snr if snr is not None else "clean",
                **metrics
            }
            rows.append(row)
        print(f'Completed {model_name}')
    return pd.DataFrame(rows)


In [ ]:
def highlight_best_worst_per_snr(df):
    """
    Bold the best metrics and italicize the worst metrics per snr_db.
    MAE/RMSE: best = min, worst = max
    SSIM: best = max, worst = min
    """
    numeric_cols = df.select_dtypes(include='number').columns
    # Identify metric types
    mae_rmse_cols = [c for c in numeric_cols if 'mae' in c.lower() or 'rmse' in c.lower()]
    ssim_cols     = [c for c in numeric_cols if 'ssim' in c.lower()]

    # Prepare empty DataFrame of styles
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    for snr, group in df.groupby('snr_db'):
        idx = group.index

        # Bold min / italic max for MAE/RMSE
        for col in mae_rmse_cols:
            min_idx = group[col].idxmin()
            max_idx = group[col].idxmax()
            styles.loc[min_idx, col] = 'font-weight: bold'
            styles.loc[max_idx, col] = 'font-style: italic'

        # Bold max / italic min for SSIM
        for col in ssim_cols:
            max_idx = group[col].idxmax()
            min_idx = group[col].idxmin()
            styles.loc[max_idx, col] = 'font-weight: bold'
            styles.loc[min_idx, col] = 'font-style: italic'

    return styles



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_metric_vs_snr(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.80):
    """
    Plot metric vs SNR for each model.

    df: DataFrame output from run_full_evaluation()
    metric_prefix: "mae", "rmse", or "ssim"
    clip_outliers: if True, y-axis will ignore extreme values based on quantiles
    lower_q, upper_q: quantiles to define the y-axis range when clipping
    """
    for mtype in ["k", "mu"]:
        plt.figure(figsize=(8,5))
        metric_col = f"{metric_prefix}_{mtype}"

        sns.lineplot(
            data=df,
            x="snr_db",
            y=metric_col,
            hue="model",
            marker="o"
        )

        plt.title(f"({mtype}) {metric_prefix.upper()} vs SNR")
        plt.xlabel("SNR (dB)")
        plt.ylabel(metric_prefix.upper())
        plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
        plt.grid(True)
        
        if clip_outliers:
            # Clip y-axis based on quantiles
            lower = df[metric_col].quantile(lower_q)
            upper = df[metric_col].quantile(upper_q)
            plt.ylim(lower, upper)

        plt.tight_layout()
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_metric_heatmap(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.75):
    """
    Plot heatmaps for a metric across models and SNR levels.
    
    df: DataFrame from run_full_evaluation()
    metric_prefix: "mae", "rmse", or "ssim"
    clip_outliers: if True, heatmap color scale will ignore extreme values
    lower_q, upper_q: quantiles for clipping when clip_outliers=True
    """
    pivot_k  = df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_k")
    pivot_mu = df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_mu")
    
    # Determine vmin/vmax if clipping
    if clip_outliers:
        k_lower = pivot_k.stack().quantile(lower_q)
        k_upper = pivot_k.stack().quantile(upper_q)
        mu_lower = pivot_mu.stack().quantile(lower_q)
        mu_upper = pivot_mu.stack().quantile(upper_q)
    else:
        k_lower, k_upper = pivot_k.min().min(), pivot_k.max().max()
        mu_lower, mu_upper = pivot_mu.min().min(), pivot_mu.max().max()
    
    fig, axes = plt.subplots(1,2, figsize=(12,6))
    
    sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[0], cmap="magma",
                vmin=k_lower, vmax=k_upper)

    axes[0].set_title(f"(k) {metric_prefix.upper()}")
    axes[0].set_xlabel("SNR (dB)")
    axes[0].set_ylabel("Model")
    
    sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[1], cmap="magma",
                vmin=mu_lower, vmax=mu_upper)
    axes[1].set_title(f"(mu) {metric_prefix.upper()}")
    axes[1].set_xlabel("SNR (dB)")
    axes[1].set_ylabel("")
    
    plt.tight_layout()
    plt.show()



In [ ]:
root = os.path.dirname(os.getcwd())
train_input = '/Training'
val_input = '/Validation'
test_input = '/Test/pt_files'
model = '/Model_Logs'
sample_pathP = root+'/MRE_PInversion/SingleSampleForCodeDebugging/pt_files/'
eval_path = root+test_input
batch_size=18
epochs=1
lr=0.001
offsets=8
fov=0.2

In [ ]:
# arch_type = 'FDTDNet'
# model, device = setup_model(arch_type, arch_subtype=None)

test_loader = get_dataloader_for_test(sample_pathP, offsets, fov, batch_size, snr_db=20)

In [ ]:
path = "/storage/coda1/p-jueda3/0/hnieves6/MRE_Inversion/Model_Logs/FDTDNet/mse/bs64weights.pth"


ckpt = torch.load(path, map_location=device)
model = setup_model("FDTDNet")[0]
model.load_state_dict(ckpt["state_dict"])


In [ ]:
res = evaluate_model_metrics(model, test_loader, device, has_inclusions=False)

In [ ]:
res

In [ ]:
models = {
    "modelA": model,
    "modelB": model,
    "modelC": model,
}
snr_levels = [5, 10, 20, 40]
run_full_evaluation(models, snr_levels, sample_pathP, device, batch_size=3, has_inclusions=False)

Testing out using multiple models

In [ ]:
base_dir = "/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs/FDTDNet/"
pth_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if "weights.pth" in file  and ("bs64" in root or "bs64" in file):
            pth_files.append(os.path.join(root, file))

print(f"Found {len(pth_files)} matching weights.pth files:")
for f in pth_files:
    print(f)

####################################################################################################

In [3]:
from pathlib import Path
import re

RUN_IDS = [
    "run_20260320_155350",
    "run_20260320_152804",
    "run_20260320_150216",
    "run_20260320_155722"
]

run_pat = re.compile(r"run_20260223_(133208|133233|140110|140733|142324|142643|173504|173414)")

# Adjust this if you want to search broader than the example path
ROOT = Path("/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs")

def bs64_has_matching_tb(bs64_dir: Path) -> bool:
    # Fast path: any directory/file path under bs64 containing the run id
    # and looks like tensorboard logs (common: events.out.tfevents*)
    for p in bs64_dir.rglob("*"):
        s = str(p)
        if run_pat.search(s) and ("tfevents" in p.name or "events.out.tfevents" in p.name):
            return True

    # Fallback: if your tb logs are stored without tfevents in filename,
    # still accept if any path under bs64 contains the run id.
    for p in bs64_dir.rglob("*"):
        if run_pat.search(str(p)):
            return True

    return False

matched_weights = []

# Find all weights.pth
for w in ROOT.rglob("weights.pth"):
    bs64_dir = w.parent  # .../bs64
    if bs64_dir.name != "bs64":
        # If sometimes the weights live deeper, you can remove this guard.
        continue

    if bs64_has_matching_tb(bs64_dir):
        matched_weights.append(str(w))

print(f"Matched weights.pth files: {len(matched_weights)}")
for p in matched_weights:
    print(p)

PermissionError: [Errno 13] Permission denied: '/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs'

In [ ]:

models = {}
for pth_path in matched_weights:
    # Make key based on relative path from base_dir to parent folder of .pth
    key = os.path.relpath(os.path.dirname(pth_path), base_dir)  # e.g., "hom/res_raw/lam0_0001"
    
    # Load checkpoint
    ckpt = torch.load(pth_path, map_location=device)
    
    # Setup model
    model = setup_model("FDTDNet")[0]  # Adjust if your setup_model returns (model, device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    model.to(device)
    
    models[key] = model

print("Loaded models:")
for k in models.keys():
    print(k)

In [ ]:
snr_levels = [5, 10, 15, 20, 25, 30]
eval_df = run_full_evaluation(models, snr_levels, sample_pathP, device, batch_size=3, has_inclusions=False)

In [ ]:
# Usage
styled_df = eval_df.sort_values('snr_db').style.apply(highlight_best_worst_per_snr, axis=None)
styled_df

In [ ]:
clip_outliers = True
plot_metric_vs_snr(eval_df, "mae", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "rmse", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "ssim", clip_outliers=False)
plt.savefig(f"{metric_prefix}_vs_snr.png", dpi=300)

In [ ]:
plot_metric_heatmap(eval_df, "rmse", clip_outliers=True)
plot_metric_heatmap(eval_df, "mae", clip_outliers=True)
plot_metric_heatmap(eval_df, "ssim", clip_outliers=False)
#plt.savefig(f"{metric_prefix}_vs_snr.png", dpi=300)

# Per sample metrics

In [ ]:
import torch
import pandas as pd
from evaluation.metrics import compute_mae, compute_rmse, compute_ssim  # your metrics.py functions

def evaluate_per_sample_metrics(model, test_loader, device, has_inclusions=False):
    """
    Evaluates model per sample and returns a long-format DataFrame.
    Includes optional CNR metric if has_inclusions=True.
    """
    model.eval()
    records = []

    with torch.no_grad():
        for wave, gt, mfre, fov, index in test_loader:
            wave, mu, gt, mfre, fov = wave.to(device), mu.to(device), gt.to(device), mfre.to(device, dtype=wave.dtype), fov.to(device, dtype=wave.dtype)
            wave = wave.to(device)
            gt   = gt.to(device)
            wave = wave.permute(4, 0, 1, 2, 3)
            wave = wave.permute(1,2,0,3,4) 
            pred_k = model(wave)
            predicted_stiffness = wave_number_to_shear_stiffness(pred_k, mfre, fov)
            gt_stiffness = mu
            

            batch_size = wave.shape[0]
            for b in range(batch_size):
                record = {
                    "index": int(index[b]),
                    "snr_db": getattr(test_loader.dataset.transform, "snr_db", None),
                    "mae_k": compute_mae(pred_k[b], gt[b]),
                    "rmse_k": compute_rmse(pred_k[b], gt[b]),
                    "ssim_k": compute_ssim(pred_k[b], gt[b]),
                    "mae_mu": compute_mae(predicted_stiffness[b], gt_stiffness[b]),
                    "rmse_mu": compute_rmse(predicted_stiffness[b], gt_stiffness[b]),
                    "ssim_mu": compute_ssim(predicted_stiffness[b], gt_stiffness[b])
                }

                # Compute CNR if inclusions are available
                if has_inclusions:
                    inc_mask, bg_mask = get_masks(gt_stiffness[b])
                    record["cnr"] = compute_cnr(predicted_stiffness[b].squeeze(), inc_mask, bg_mask).item()

                records.append(record)

    return pd.DataFrame(records)


import pandas as pd
from torch.utils.data import DataLoader

def run_full_evaluation_per_sample(
    models,           # dict: name -> model
    snr_levels,       # list of SNR values (or None for clean)
    test_dir,         # path to test data
    device="cuda",    # device for inference
    batch_size=1,     # dataloader batch size
    has_inclusions=False
):
    """
    Sweep over multiple models and SNR levels, returning per-sample metrics.
    Returns a long-format pandas DataFrame with one row per test item.
    """
    all_rows = []

    for model_name, model in models.items():
        for snr in snr_levels:
            loader = get_dataloader_for_test(
                test_dir,
                batch_size=batch_size,
                snr_db=snr
            )

            # Use the per-sample evaluation function
            df_sample_metrics = evaluate_per_sample_metrics(
                model,
                loader,
                device,
                has_inclusions=has_inclusions
            )

            # Add metadata columns
            df_sample_metrics["model"] = model_name
            df_sample_metrics["snr_db"] = snr if snr is not None else "clean"

            all_rows.append(df_sample_metrics)
        print(f'Completed {model_name}')

    # Concatenate all results into one DataFrame
    return pd.concat(all_rows, ignore_index=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_per_sample_metrics(df, metrics=None, plot_type="box", figsize=(12,6), clip=None):
    """
    Plot distributions of per-sample metrics across models and SNRs, with optional clipping.

    Args:
        df (pd.DataFrame): Long-format per-sample metrics DataFrame from evaluate_per_sample_metrics.
        metrics (list): List of metrics to plot. Default: all numeric metrics.
        plot_type (str): 'box' or 'violin'.
        figsize (tuple): Figure size.
        clip (tuple or None): Optional (min, max) to clip y-axis values.
    """
    if metrics is None:
        # Automatically pick numeric columns
        metrics = [c for c in df.columns if c not in ["index", "snr_db", "model"]]

    for metric in metrics:
        plt.figure(figsize=figsize)
        plot_data = df.copy()

        # Clip metric values if requested
        if clip is not None:
            min_val, max_val = clip
            plot_data[metric] = plot_data[metric].clip(lower=min_val, upper=max_val)

        if plot_type == "box":
            sns.boxplot(
                data=plot_data,
                x="snr_db",
                y=metric,
                hue="model",
                palette="Set2"
            )
        elif plot_type == "violin":
            sns.violinplot(
                data=plot_data,
                x="snr_db",
                y=metric,
                hue="model",
                split=True,   # split violin per model
                palette="Set2"
            )
        else:
            raise ValueError("plot_type must be 'box' or 'violin'")

        plt.title(f"Per-sample {metric} distribution across models and SNR")
        plt.xlabel("SNR (dB)")
        plt.ylabel(metric)
        plt.legend(title="Model")
        plt.tight_layout()
        plt.show()



In [2]:
snr_levels = [5, 10, 15, 20, 25, 30]
per_sample_df = run_full_evaluation_per_sample(models, snr_levels, sample_pathP, device, batch_size=3, has_inclusions=False)

NameError: name 'run_full_evaluation_per_sample' is not defined

In [3]:
per_sample_df

NameError: name 'per_sample_df' is not defined

In [4]:
plot_per_sample_metrics(per_sample_df, metrics=["mae_k", "rmse_k", "ssim_k"], plot_type="box", clip = (0,12))

NameError: name 'plot_per_sample_metrics' is not defined

# Run on Test Dataset

In [5]:
root = os.path.dirname(os.getcwd())
train_input = '/Training'
val_input = '/Validation'
test_input = '/Test/pt_files'
model = '/Model_Logs'
inclusion = '/Inclusion'
sample_pathP = root+'/MRE_PInversion/SingleSampleForCodeDebugging/pt_files/'
eval_path = root+test_input
inclusion_path = root+inclusion
batch_size=64
epochs=1
lr=0.001
offsets=8
fov=0.2

In [6]:
from pathlib import Path
import re

RUN_IDS = [
    "run_20260223_133208",
    "run_20260223_133233",
    "run_20260223_140110",
    "run_20260223_140733",
    "run_20260223_142324",
    "run_20260223_142643",
    "run_20260223_173504",
    "run_20260223_173414",
]

run_pat = re.compile(r"run_20260223_(133208|133233|140110|140733|142324|142643|173504|173414)")

# Adjust this if you want to search broader than the example path
ROOT = Path("/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs")

def bs64_has_matching_tb(bs64_dir: Path) -> bool:
    # Fast path: any directory/file path under bs64 containing the run id
    # and looks like tensorboard logs (common: events.out.tfevents*)
    for p in bs64_dir.rglob("*"):
        s = str(p)
        if run_pat.search(s) and ("tfevents" in p.name or "events.out.tfevents" in p.name):
            return True

    # Fallback: if your tb logs are stored without tfevents in filename,
    # still accept if any path under bs64 contains the run id.
    for p in bs64_dir.rglob("*"):
        if run_pat.search(str(p)):
            return True

    return False

matched_weights = []

# Find all weights.pth
for w in ROOT.rglob("weights.pth"):
    bs64_dir = w.parent  # .../bs64
    if bs64_dir.name != "bs64":
        # If sometimes the weights live deeper, you can remove this guard.
        continue

    if bs64_has_matching_tb(bs64_dir):
        matched_weights.append(str(w))

print(f"Matched weights.pth files: {len(matched_weights)}")
for p in matched_weights:
    print(p)
    
models = {}
for pth_path in matched_weights:
    # Make key based on relative path from base_dir to parent folder of .pth
    key = os.path.relpath(os.path.dirname(pth_path), base_dir)  # e.g., "hom/res_raw/lam0_0001"
    
    # Load checkpoint
    ckpt = torch.load(pth_path, map_location=device)
    
    # Setup model
    model = setup_model("FDTDNet")[0]  # Adjust if your setup_model returns (model, device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    model.to(device)
    
    models[key] = model

print("Loaded models:")
for k in models.keys():
    print(k)

PermissionError: [Errno 13] Permission denied: '/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs'

In [ ]:
base_dir = "/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Model_Logs/FDTDNet/residual"
pth_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if "weights.pth" in file  and ("bs64" in root or "bs64" in file):
            pth_files.append(os.path.join(root, file))

print(f"Found {len(pth_files)} matching weights.pth files:")
for f in pth_files:
    print(f)

####################################################################################################

models = {}
for pth_path in pth_files:
    # Make key based on relative path from base_dir to parent folder of .pth
    key = os.path.relpath(os.path.dirname(pth_path), base_dir)  # e.g., "hom/res_raw/lam0_0001"
    
    # Load checkpoint
    ckpt = torch.load(pth_path, map_location=device)
    
    # Setup model
    model = setup_model("FDTDNet")[0]  # Adjust if your setup_model returns (model, device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    model.to(device)
    
    models[key] = model

print("Loaded models:")
for k in models.keys():
    print(k)

In [7]:
snr_levels = [15, 20, 25, 30]
#uncomment below if u want a specific model
#key = list(models.keys())[5]
#single_model_dict = {key: models[key]}
eval_df = run_full_evaluation(models, snr_levels, eval_path, device, batch_size=64, has_inclusions=False)

NameError: name 'run_full_evaluation' is not defined

In [8]:
styled_df = eval_df.sort_values('snr_db').style.apply(highlight_best_worst_per_snr, axis=None)
styled_df

NameError: name 'eval_df' is not defined

In [9]:
clip_outliers = False
plot_metric_vs_snr(eval_df, "mae", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "rmse", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "ssim", clip_outliers=False)
#plt.savefig(f"{metric_prefix}_vs_snr.png", dpi=300)

plot_metric_heatmap(eval_df, "rmse", clip_outliers=clip_outliers)
plot_metric_heatmap(eval_df, "mae", clip_outliers=clip_outliers)
plot_metric_heatmap(eval_df, "ssim", clip_outliers=False)

NameError: name 'plot_metric_vs_snr' is not defined

In [10]:
clip_outliers = True
plot_metric_vs_snr(eval_df, "mae", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "rmse", clip_outliers=clip_outliers)
plot_metric_vs_snr(eval_df, "ssim", clip_outliers=False)
#plt.savefig(f"{metric_prefix}_vs_snr.png", dpi=300)

plot_metric_heatmap(eval_df, "rmse", clip_outliers=clip_outliers)
plot_metric_heatmap(eval_df, "mae", clip_outliers=clip_outliers)
plot_metric_heatmap(eval_df, "ssim", clip_outliers=False)

NameError: name 'plot_metric_vs_snr' is not defined

In [11]:
snr_levels = [15, 20, 25, 30]
per_sample_df = run_full_evaluation_per_sample(models, snr_levels, eval_path, device, batch_size=64, has_inclusions=False)

NameError: name 'run_full_evaluation_per_sample' is not defined

In [12]:
per_sample_df

NameError: name 'per_sample_df' is not defined

In [13]:
from datetime import date

base_dir = "evaluation/results"
per_sample_dir = os.path.join(base_dir, "per_sample")
aggregate_dir = os.path.join(base_dir, "global")

os.makedirs(per_sample_dir, exist_ok=True)
os.makedirs(aggregate_dir, exist_ok=True)

out_dir = "evaluation/results"
os.makedirs(out_dir, exist_ok=True)


# ---- per-sample metrics (PARQUET) ----
per_sample_fname = (
    f"per_sample_metrics__"
    f"models-{len(models)}__"
    f"snr-{snr_levels}__"
    f"{date.today()}.parquet"
)

per_sample_df.to_parquet(
    os.path.join(per_sample_dir, per_sample_fname),
    index=False
)

# ---- global / aggregate metrics (CSV) ----
aggregate_fname = (
    f"global_eval_metrics__"
    f"models-{len(models)}__"
    f"snr-{snr_levels}__"
    f"{date.today()}.csv"
)

eval_df.to_csv(
    os.path.join(aggregate_dir, aggregate_fname),
    index=False
)

NameError: name 'models' is not defined

In [14]:
plot_per_sample_metrics(per_sample_df, metrics=["mae_k", "rmse_k", "ssim_k"], plot_type="box", clip = (0,12))

NameError: name 'plot_per_sample_metrics' is not defined

In [15]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

def plot_gt_pred_error_across_snr_models(
    models,
    snr_levels,
    test_dir,
    device,
    param="mu",           # "mu" or "k"
    num_samples=5,
    batch_size=1,
    vmin=None,
    vmax=None,
    err_vmax=None,
    save_dir=None
):
    """
    Plots GT | pred @ high SNR → low SNR
    Each sample gets its own separate figure with:
    Row 1: images (GT + predictions)
    Row 2: error maps
    """

    snr_levels = sorted(snr_levels, reverse=True)

    for model_name, model in models.items():
        print(f"Plotting model: {model_name}")

        # -------------------------------
        # Grab first `num_samples` from clean loader to fix indices
        # -------------------------------
        loader_clean = get_dataloader_for_test(test_dir, batch_size=1, snr_db=None)
        fixed_samples = []
        for i, (wave, mu, gt, mfre, fov, idx) in enumerate(loader_clean):
            if i >= num_samples:
                break
            fixed_samples.append((wave, mu, gt, mfre, fov, int(idx[0])))

        # -------------------------------
        # Process each sample separately
        # -------------------------------
        for sample_idx, (wave, mu, gt, mfre, fov, idx) in enumerate(fixed_samples):
            
            ncols = 1 + len(snr_levels)  # GT + SNR predictions
            nrows = 2  # Row 1: images, Row 2: errors

            # Create a separate figure for this sample with gridspec for better control
            fig = plt.figure(figsize=(3*ncols + 1, 6))  # Extra space for colorbars
            gs = fig.add_gridspec(nrows, ncols, hspace=0.05, wspace=0.05)
            axes = np.array([[fig.add_subplot(gs[i, j]) for j in range(ncols)] for i in range(nrows)])

            # Move tensors to device
            wave = wave.to(device)
            gt   = gt.to(device)
            mu = mu.to(device)
            mfre = mfre.to(device, dtype=wave.dtype)
            fov  = fov.to(device, dtype=wave.dtype)

            # ✅ FIX: Prepare GT image based on param
            if param == "mu":
                gt_img = mu.squeeze().cpu().numpy() / 1000.0  # Convert Pa → kPa for display
            else:  # param == "k"
                gt_img = gt.squeeze().cpu().numpy()  # Use gt (wave number)
            
            # ✅ FIX: Check if gt_img has variation
            print(f"Sample {idx}: GT {param} range = [{gt_img.min():.2f}, {gt_img.max():.2f}], mean = {gt_img.mean():.2f}")

            # Plot GT (row 0, col 0)
            im_gt = axes[0, 0].imshow(gt_img, cmap="viridis", vmin=vmin, vmax=vmax)
            axes[0, 0].set_title("GT")
            axes[0, 0].set_ylabel(f"Sample {idx}")
            axes[0, 0].axis("off")

            # Plot dummy error for GT column (row 1, col 0)
            im_err = axes[1, 0].imshow(np.zeros_like(gt_img), cmap="seismic", vmin=-err_vmax, vmax=err_vmax)
            axes[1, 0].axis("off")

            # -------------------------------
            # Plot predictions and errors for each SNR level
            # -------------------------------
            for col_idx, snr in enumerate(snr_levels):
                loader_snr = get_dataloader_for_test(test_dir, batch_size=1, snr_db=snr)

                # Find matching sample by index
                for w, mu_snr, g, mf, fv, ix in loader_snr:
                    if int(ix[0]) == idx:
                        w = w.to(device)
                        mf = mf.to(device, dtype=w.dtype)
                        fv = fv.to(device, dtype=w.dtype)
                        w_model = w.permute(4,0,1,2,3).permute(1,2,0,3,4)
                        
                        with torch.no_grad():
                            pred_k = model(w_model)
                        
                        # ✅ FIX: Convert prediction based on param
                        if param == "mu":
                            pred_img = wave_number_to_shear_stiffness(pred_k, mf, fv).squeeze().cpu().numpy() / 1000.0  # Pa → kPa
                        else:  # param == "k"
                            pred_img = pred_k.squeeze().cpu().numpy()
                        break

                # Prediction plot (row 0)
                axes[0, col_idx+1].imshow(pred_img, cmap="viridis", vmin=vmin, vmax=vmax)
                axes[0, col_idx+1].set_title(f"SNR {snr}")
                axes[0, col_idx+1].axis("off")

                # Error plot (row 1) - all use same colorbar range
                err_img = pred_img - gt_img
                axes[1, col_idx+1].imshow(err_img, cmap="seismic", vmin=-err_vmax, vmax=err_vmax)
                axes[1, col_idx+1].axis("off")

            # -------------------------------
            # Add colorbars on the left side without shrinking the axes
            # -------------------------------
            # Get positions for colorbar placement
            pos_gt = axes[0, 0].get_position()
            pos_err = axes[1, 0].get_position()
            
            # Create colorbar axes manually to the left
            cbar_ax_gt = fig.add_axes([pos_gt.x0 - 0.05, pos_gt.y0, 0.02, pos_gt.height])
            cbar_gt = fig.colorbar(im_gt, cax=cbar_ax_gt)
            cbar_gt.set_label(f"{param} (kPa)" if param == "mu" else param)

            cbar_ax_err = fig.add_axes([pos_err.x0 - 0.05, pos_err.y0, 0.02, pos_err.height])
            cbar_err = fig.colorbar(im_err, cax=cbar_ax_err)
            cbar_err.set_label("Error")

            # Add a main title for the figure
            fig.suptitle(f"Model: {model_name} | Sample {idx}", fontsize=14, y=0.98)

            # -------------------------------
            # Save figure if requested
            # -------------------------------
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)
                fpath = os.path.join(save_dir, f"{model_name.replace('/','_')}_sample_{idx}_gt_pred_error.png")
                plt.savefig(fpath, dpi=200, bbox_inches='tight')
                print(f"Saved: {fpath}")

            plt.show()
            plt.close(fig)  # Close figure to free memory

In [16]:
plot_gt_pred_error_across_snr_models(
    models=models,
    snr_levels=[30, 25, 20, 15],
    test_dir=sample_pathP,
    device=device,
    param="mu",
    num_samples=5,
    vmin=0,
    vmax=15,
    err_vmax=15.0,
)

NameError: name 'models' is not defined

# Load evaluation Results

In [17]:
import pandas as pd
import os

per_sample_path = "evaluation/results/per_sample/per_sample_metrics__models-11__snr-[15, 20, 25, 30]__2026-02-02.parquet"
aggregate_path  = "evaluation/results/global/global_eval_metrics__models-11__snr-[15, 20, 25, 30]__2026-02-02.csv"
per_sample_df = pd.read_parquet(per_sample_path)
global_df = pd.read_csv(aggregate_path)

print(per_sample_df.shape)
print(global_df.shape)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

# Inclusion Results

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def plot_per_sample_metrics(df, metrics=None, plot_type="box", figsize=(12,6), clip=None, dataset_filter=None, show=False):
    """
    Plot distributions of per-sample metrics across models and SNRs, with optional clipping.
    
    Args:
        df (pd.DataFrame): Long-format per-sample metrics DataFrame.
        metrics (list): List of metrics to plot. Default: all numeric metrics.
        plot_type (str): 'box' or 'violin'.
        figsize (tuple): Figure size.
        clip (tuple or None): Optional (min, max) to clip y-axis values.
        dataset_filter (str or None): Filter to specific dataset (e.g., "Main_Test", "Inclusion_R0.3125cm")
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    if metrics is None:
        # Automatically pick numeric columns
        metrics = [c for c in plot_df.columns if c not in ["index", "snr_db", "model", "dataset"]]
    
    for metric in metrics:
        plt.figure(figsize=figsize)
        
        # Clip metric values if requested
        if clip is not None:
            min_val, max_val = clip
            plot_df[metric] = plot_df[metric].clip(lower=min_val, upper=max_val)
        
        if plot_type == "box":
            sns.boxplot(
                data=plot_df,
                x="snr_db",
                y=metric,
                hue="model",
                palette="Set2"
            )
        elif plot_type == "violin":
            sns.violinplot(
                data=plot_df,
                x="snr_db",
                y=metric,
                hue="model",
                split=False,  # Changed to False for multiple models
                palette="Set2"
            )
        else:
            raise ValueError("plot_type must be 'box' or 'violin'")
        
        title = f"Per-sample {metric} distribution"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel(metric)
        plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        if show:
            plt.show()


def plot_metric_heatmap(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.75, dataset_filter=None, show=False):
    """
    Plot heatmaps for a metric across models and SNR levels.
    
    Args:
        df: DataFrame from run_full_evaluation() (aggregate)
        metric_prefix: "mae", "rmse", "ssim", or "cnr"
        clip_outliers: if True, heatmap color scale will ignore extreme values
        lower_q, upper_q: quantiles for clipping when clip_outliers=True
        dataset_filter: Filter to specific dataset
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    # Check if metric has _k and _mu variants
    has_k = f"{metric_prefix}_k" in plot_df.columns
    has_mu = f"{metric_prefix}_mu" in plot_df.columns
    is_cnr = metric_prefix == "cnr" and "cnr" in plot_df.columns
    
    if is_cnr:
        # CNR only has one column
        pivot = plot_df.pivot(index="model", columns="snr_db", values="cnr")
        
        if clip_outliers:
            vmin = pivot.stack().quantile(lower_q)
            vmax = pivot.stack().quantile(upper_q)
        else:
            vmin, vmax = pivot.min().min(), pivot.max().max()
        
        plt.figure(figsize=(8,6))
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=vmin, vmax=vmax)
        title = f"CNR"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel("Model")
        plt.tight_layout()
        plt.show()
        
    elif has_k and has_mu:
        pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_k")
        pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric_prefix}_mu")
        
        # Determine vmin/vmax if clipping
        if clip_outliers:
            k_lower = pivot_k.stack().quantile(lower_q)
            k_upper = pivot_k.stack().quantile(upper_q)
            mu_lower = pivot_mu.stack().quantile(lower_q)
            mu_upper = pivot_mu.stack().quantile(upper_q)
        else:
            k_lower, k_upper = pivot_k.min().min(), pivot_k.max().max()
            mu_lower, mu_upper = pivot_mu.min().min(), pivot_mu.max().max()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[0], cmap="magma",
                    vmin=k_lower, vmax=k_upper)
        axes[0].set_title(f"(k) {metric_prefix.upper()}")
        axes[0].set_xlabel("SNR (dB)")
        axes[0].set_ylabel("Model")
        
        sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[1], cmap="magma",
                    vmin=mu_lower, vmax=mu_upper)
        axes[1].set_title(f"(mu) {metric_prefix.upper()}")
        axes[1].set_xlabel("SNR (dB)")
        axes[1].set_ylabel("")
        
        if dataset_filter:
            fig.suptitle(dataset_filter, fontsize=14, y=1.02)
        
        plt.tight_layout()
        if show:
            plt.show()


def plot_metric_vs_snr(df, metric_prefix="mae", clip_outliers=False, lower_q=0, upper_q=0.80, dataset_filter=None, show=False):
    """
    Plot metric vs SNR for each model.
    
    Args:
        df: DataFrame output from run_full_evaluation() (aggregate)
        metric_prefix: "mae", "rmse", "ssim", or "cnr"
        clip_outliers: if True, y-axis will ignore extreme values based on quantiles
        lower_q, upper_q: quantiles to define the y-axis range when clipping
        dataset_filter: Filter to specific dataset
    """
    # Filter by dataset if specified
    plot_df = df[df["dataset"] == dataset_filter].copy() if dataset_filter else df.copy()
    
    # Convert snr_db to numeric for proper sorting and plotting
    snr_mapping = {"clean": 999}
    plot_df["snr_numeric"] = plot_df["snr_db"].apply(
        lambda x: snr_mapping.get(x, float(x)) if x != "clean" else 999
    )
    plot_df = plot_df.sort_values("snr_numeric")
    
    # Create x-axis labels (keep original labels)
    plot_df["snr_label"] = plot_df["snr_db"].apply(
        lambda x: "Clean" if x == "clean" else str(x)
    )
    
    is_cnr = metric_prefix == "cnr" and "cnr" in plot_df.columns
    
    if is_cnr:
        plt.figure(figsize=(8, 5))
        sns.lineplot(
            data=plot_df,
            x="snr_numeric",  # Use numeric for plotting
            y="cnr",
            hue="model",
            marker="o",
            style="model"
        )
        
        # Fix x-axis labels
        unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
        plt.xticks(unique_snr["snr_numeric"], unique_snr["snr_label"])
        
        title = "CNR vs SNR"
        if dataset_filter:
            title += f" - {dataset_filter}"
        plt.title(title)
        plt.xlabel("SNR (dB)")
        plt.ylabel("CNR")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.grid(True)
        
        if clip_outliers:
            lower = plot_df["cnr"].quantile(lower_q)
            upper = plot_df["cnr"].quantile(upper_q)
            plt.ylim(lower, upper)
        
        plt.tight_layout()
        plt.show()
    else:
        for mtype in ["k", "mu"]:
            metric_col = f"{metric_prefix}_{mtype}"
            if metric_col not in plot_df.columns:
                continue
                
            plt.figure(figsize=(8, 5))
            sns.lineplot(
                data=plot_df,
                x="snr_numeric",  # Use numeric for plotting
                y=metric_col,
                hue="model",
                marker="o",
                style="model"
            )
            
            # Fix x-axis labels
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            plt.xticks(unique_snr["snr_numeric"], unique_snr["snr_label"])
            
            title = f"({mtype}) {metric_prefix.upper()} vs SNR"
            if dataset_filter:
                title += f" - {dataset_filter}"
            plt.title(title)
            plt.xlabel("SNR (dB)")
            plt.ylabel(metric_prefix.upper())
            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            plt.grid(True)
            
            if clip_outliers:
                lower = plot_df[metric_col].quantile(lower_q)
                upper = plot_df[metric_col].quantile(upper_q)
                plt.ylim(lower, upper)
            
            plt.tight_layout()
            if show:
                plt.show()


def highlight_best_per_snr_dataset(df):
    """
    Bold the best metrics per SNR and dataset in aggregate DataFrame.
    Returns a styled DataFrame.
    
    For MAE/RMSE: best = min (bold green)
    For SSIM/CNR: best = max (bold green)
    """
    # Identify metric columns
    numeric_cols = df.select_dtypes(include='number').columns
    mae_rmse_cols = [c for c in numeric_cols if 'mae' in c.lower() or 'rmse' in c.lower()]
    ssim_cnr_cols = [c for c in numeric_cols if 'ssim' in c.lower() or 'cnr' in c.lower()]
    
    def apply_highlighting(data):
        """Apply styles to entire DataFrame at once."""
        # Create empty style DataFrame
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        
        # Group by dataset and snr_db
        for (dataset, snr), group_idx in data.groupby(['dataset', 'snr_db']).groups.items():
            # Get the subset of data for this group
            group_data = data.loc[group_idx]
            
            # Highlight best for MAE/RMSE (minimum)
            for col in mae_rmse_cols:
                if col in group_data.columns and not group_data[col].isna().all():
                    best_idx = group_data[col].idxmin()
                    styles.loc[best_idx, col] = 'font-weight: bold; color: green'
            
            # Highlight best for SSIM/CNR (maximum)
            for col in ssim_cnr_cols:
                if col in group_data.columns and not group_data[col].isna().all():
                    best_idx = group_data[col].idxmax()
                    styles.loc[best_idx, col] = 'font-weight: bold; color: green'
        
        return styles
    
    return df.style.apply(apply_highlighting, axis=None)


def display_best_metrics_table(df, dataset_filter=None, snr_filter=None):
    """
    Display a clean summary table showing only the best model per metric.
    
    Args:
        df: Aggregate metrics DataFrame
        dataset_filter: Filter to specific dataset
        snr_filter: Filter to specific SNR value
    """
    filtered_df = df.copy()
    
    if dataset_filter:
        filtered_df = filtered_df[filtered_df['dataset'] == dataset_filter]
    if snr_filter is not None:
        filtered_df = filtered_df[filtered_df['snr_db'] == snr_filter]
    
    # Get metric columns
    numeric_cols = filtered_df.select_dtypes(include='number').columns
    mae_rmse_cols = [c for c in numeric_cols if 'mae' in c.lower() or 'rmse' in c.lower()]
    ssim_cnr_cols = [c for c in numeric_cols if 'ssim' in c.lower() or 'cnr' in c.lower()]
    
    results = []
    
    for (dataset, snr), group in filtered_df.groupby(['dataset', 'snr_db']):
        row = {'dataset': dataset, 'snr_db': snr}
        
        # Find best model for each metric (skip if all NaN)
        for col in mae_rmse_cols:
            if col in group.columns and not group[col].isna().all():
                best_idx = group[col].idxmin()
                if pd.notna(best_idx):  # Check if index is valid
                    best_model = group.loc[best_idx, 'model']
                    best_value = group.loc[best_idx, col]
                    row[f'{col}_model'] = best_model
                    row[f'{col}_value'] = f'{best_value:.4f}'
        
        for col in ssim_cnr_cols:
            if col in group.columns and not group[col].isna().all():
                best_idx = group[col].idxmax()
                if pd.notna(best_idx):  # Check if index is valid
                    best_model = group.loc[best_idx, 'model']
                    best_value = group.loc[best_idx, col]
                    row[f'{col}_model'] = best_model
                    row[f'{col}_value'] = f'{best_value:.4f}'
        
        results.append(row)
    
    summary_df = pd.DataFrame(results)
    return summary_df

In [5]:
import os
import re
import torch
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional
from Data_loader import get_dataloader_for_test
from train_functions import wave_number_to_shear_stiffness, setup_model
from evaluation.metrics import compute_mae, compute_rmse, compute_ssim, compute_cnr, get_masks
from datetime import datetime
from pathlib import Path
import json


class ModelEvaluator:
    """Comprehensive model evaluation across datasets and SNR levels."""
    
    def __init__(
        self,
        root_dir: str = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs",
        device: str = "cuda"
    ):
        self.root = Path(root_dir)
        self.device = device
        
    def find_weights_by_runs(self, run_ids: List[str]) -> List[str]:
        """
        Find all weights.pth matching specific run IDs.
        
        Args:
            run_ids: List of run IDs like ["run_20260223_133208", "run_20260223_140110"]
        """
        # Extract just the numeric parts if full run_YYYYMMDD_HHMMSS provided
        numeric_ids = []
        for rid in run_ids:
            if rid.startswith("run_"):
                # Extract YYYYMMDD_HHMMSS part
                parts = rid.split("_")
                if len(parts) >= 3:
                    numeric_ids.append(parts[-1])  # Get HHMMSS
                else:
                    numeric_ids.append(rid)
            else:
                numeric_ids.append(rid)
        
        # Build regex pattern
        run_pat = re.compile(r"run_\d{8}_(" + "|".join(numeric_ids) + r")")
        
        def bs64_has_matching_tb(bs64_dir: Path) -> bool:
            """Check if bs64 directory contains matching tensorboard logs."""
            # Fast path: look for tfevents files with matching run id
            for p in bs64_dir.rglob("*"):
                s = str(p)
                if run_pat.search(s) and ("tfevents" in p.name or "events.out.tfevents" in p.name):
                    return True
            
            # Fallback: any path under bs64 contains the run id
            for p in bs64_dir.rglob("*"):
                if run_pat.search(str(p)):
                    return True
            
            return False
        
        matched_weights = []
        
        # Find all weights.pth
        for w in self.root.rglob("weights.pth"):
            bs64_dir = w.parent  # .../bs64
            if bs64_dir.name != "bs64":
                # If weights live deeper, you can remove this guard
                continue
            
            if bs64_has_matching_tb(bs64_dir):
                matched_weights.append(str(w))
        
        print(f"Matched weights.pth files: {len(matched_weights)}")
        for p in matched_weights:
            print(f"  {p}")
        
        return matched_weights
    
    def find_weights_by_filter(self, filter_dict: Dict[str, any]) -> List[str]:
        """
        Find weights.pth based on flexible filters.
        
        filter_dict examples:
            {"loss_type": "mse"}
            {"loss_type": "residual", "lambda_physics": 0.025}
            {"loss_type": ["mse", "residual"]}
        """
        matched_weights = []
        
        for w in self.root.rglob("weights.pth"):
            # Load checkpoint to check metadata
            try:
                ckpt = torch.load(w, map_location="cpu")
                
                # Check all filter conditions
                match = True
                for key, value in filter_dict.items():
                    if key not in ckpt:
                        match = False
                        break
                    
                    # Handle list of acceptable values
                    if isinstance(value, list):
                        if ckpt[key] not in value:
                            match = False
                            break
                    else:
                        if ckpt[key] != value:
                            match = False
                            break
                
                if match:
                    matched_weights.append(str(w))
                    
            except Exception as e:
                print(f"Warning: Could not load {w}: {e}")
                continue
        
        print(f"Found {len(matched_weights)} weights matching filter: {filter_dict}")
        return matched_weights
    
    def load_models(self, weight_paths: List[str]) -> Dict[str, torch.nn.Module]:
        """Load models from weight paths."""
        models = {}
        
        for pth_path in weight_paths:
            # Create key based on relative path from root
            path_obj = Path(pth_path)
            key = os.path.relpath(os.path.dirname(pth_path), self.root)
            
            # Load checkpoint
            ckpt = torch.load(pth_path, map_location=self.device)
            
            # Setup model
            model = setup_model("FDTDNet")[0]
            model.load_state_dict(ckpt["state_dict"])
            model.eval()
            model.to(self.device)
            
            models[key] = model
        
        print(f"\nLoaded {len(models)} models:")
        for k in models.keys():
            print(f"  - {k}")
        
        return models
    
    def evaluate_per_sample_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> pd.DataFrame:
        """Evaluate model per sample and return DataFrame."""
        model.eval()
        records = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                # Permute wave
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                # Predict
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                # Metrics per sample
                batch_size = wave.shape[0]
                for b in range(batch_size):
                    record = {
                        "index": int(index[b]),
                        "mae_k": compute_mae(pred_k[b], k_gt[b]),
                        "rmse_k": compute_rmse(pred_k[b], k_gt[b]),
                        "ssim_k": compute_ssim(pred_k[b], k_gt[b]),
                        "mae_mu": compute_mae(pred_mu[b], mu[b]),
                        "rmse_mu": compute_rmse(pred_mu[b], mu[b]),
                        "ssim_mu": compute_ssim(pred_mu[b], mu[b])
                    }
                    
                    # CNR if inclusions
                    if has_inclusions:
                        inc_mask, bg_mask = get_masks(mu[b])
                        cnr_value = compute_cnr(
                        pred_mu[b].squeeze(),
                        inc_mask,
                        bg_mask
                        )
                        record["cnr"] = cnr_value.item() if torch.is_tensor(cnr_value) else cnr_value
                    records.append(record)  # ← This line was missing!

        
        return pd.DataFrame(records)
    
    def evaluate_aggregate_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> Dict:
        """Evaluate model and return aggregated metrics."""
        model.eval()
        metrics = {
            "mae_k": [], "rmse_k": [], "ssim_k": [],
            "mae_mu": [], "rmse_mu": [], "ssim_mu": []
        }
        if has_inclusions:
            metrics["cnr"] = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                metrics["mae_k"].append(compute_mae(pred_k, k_gt))
                metrics["rmse_k"].append(compute_rmse(pred_k, k_gt))
                metrics["ssim_k"].append(compute_ssim(pred_k, k_gt))
                metrics["mae_mu"].append(compute_mae(pred_mu, mu))
                metrics["rmse_mu"].append(compute_rmse(pred_mu, mu))
                metrics["ssim_mu"].append(compute_ssim(pred_mu, mu))
                
                if has_inclusions:
                    inc_mask, bg_mask = get_masks(mu)
                    metrics["cnr"].append(compute_cnr(pred_mu.squeeze(), inc_mask, bg_mask))
        
        return {k: (torch.tensor(v).mean().item() if v else 0.0) for k, v in metrics.items()}
    
    def run_full_evaluation(
        self,
        models: Dict[str, torch.nn.Module],
        test_configs: List[Dict],
        per_sample: bool = False
    ) -> pd.DataFrame:
        """
        Run evaluation across models and test configurations.
        
        test_configs: List of dicts with keys:
            - test_dir: path to test data
            - snr_levels: list of SNR values (None for clean)
            - has_inclusions: bool
            - dataset_name: str (for labeling)
        """
        all_rows = []
        
        for config in test_configs:
            test_dir = config["test_dir"]
            snr_levels = config["snr_levels"]
            has_inclusions = config.get("has_inclusions", False)
            dataset_name = config.get("dataset_name", "Unknown")
            
            print(f"\n{'='*60}")
            print(f"Evaluating on: {dataset_name}")
            print(f"{'='*60}")
            
            for model_name, model in models.items():
                for snr in snr_levels:
                    loader = get_dataloader_for_test(
                        test_dir,
                        batch_size=1,
                        snr_db=snr
                    )
                    
                    if per_sample:
                        df = self.evaluate_per_sample_metrics(
                            model, loader, has_inclusions
                        )
                        df["model"] = model_name
                        df["snr_db"] = snr if snr is not None else "clean"
                        df["dataset"] = dataset_name
                        all_rows.append(df)
                    else:
                        metrics = self.evaluate_aggregate_metrics(
                            model, loader, has_inclusions
                        )
                        row = {
                            "model": model_name,
                            "snr_db": snr if snr is not None else "clean",
                            "dataset": dataset_name,
                            **metrics
                        }
                        all_rows.append(pd.DataFrame([row]))
                
                print(f"  ✓ Completed {model_name}")
        
        return pd.concat(all_rows, ignore_index=True)


# ==================== USAGE EXAMPLE ====================

def main():
    # Initialize evaluator
    evaluator = ModelEvaluator(device="cuda")
    
    # Find by run IDs (exact format from your example)
    RUN_IDS = [
        "run_20260311_235821",
        "run_20260312_112910",
        "run_20260312_112911",
        "run_20260312_114204",
    ]
    
    weight_paths = evaluator.find_weights_by_runs(RUN_IDS)
    
    # Load models
    models = evaluator.load_models(weight_paths)
    
    # Define test configurations
    base_test_dir = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Data"
    snr_levels = [30, 25, 20, 15]
    
    test_configs = [
        # Main test set
        {
            "test_dir": f"{base_test_dir}/Test/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": False,
            "dataset_name": "Main_Test"
        },
        # Inclusion datasets
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_3125cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.3125cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_625cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.625cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R1_25cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R1.25cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R2_5cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R2.5cm"
        }
    ]
    
    # Run evaluations
    print("\n" + "="*60)
    print("AGGREGATE METRICS EVALUATION")
    print("="*60)
    df_aggregate = evaluator.run_full_evaluation(
        models, test_configs, per_sample=False
    )
    
    print("\n" + "="*60)
    print("PER-SAMPLE METRICS EVALUATION")
    print("="*60)
    df_per_sample = evaluator.run_full_evaluation(
        models, test_configs, per_sample=True
    )
    
    # Save results
    # Create timestamped folder
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path("evaluation_results") / timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create subfolder for plots
    plots_dir = output_dir / "plots"
    plots_dir.mkdir(exist_ok=True)

    # Save CSVs
    df_aggregate.to_csv(output_dir / "aggregate_metrics.csv", index=False)
    df_per_sample.to_csv(output_dir / "per_sample_metrics.csv", index=False)

#     # Save styled Excel with highlighted best values
#     styled_df = highlight_best_per_snr_dataset(df_aggregate)
#     styled_df.to_excel(
#         output_dir / "aggregate_highlighted.xlsx",
#         engine="openpyxl",
#         index=False
#     )

    # Save summary of best models per metric
    summary = display_best_metrics_table(df_aggregate)
    summary.to_csv(output_dir / "best_models_summary.csv", index=False)
    
    # Generate and save plots for each dataset
    print("\nGenerating plots...")
    datasets = df_aggregate["dataset"].unique()

    for dataset in datasets:
        print(f"  Plotting {dataset}...")
        dataset_safe = dataset.replace("/", "_").replace(" ", "_").replace('.','_')

        # Determine metrics (add CNR if inclusions)
        metrics = ["mae", "rmse", "ssim"]
        has_cnr = "Inclusion" in dataset and "cnr" in df_aggregate.columns
        if has_cnr:
            metrics.append("cnr")

        # ========== COMBINED HEATMAP FIGURE ==========
        n_metrics = len(metrics)
        fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
        if n_metrics == 1:
            axes = axes.reshape(1, -1)

        for idx, metric in enumerate(metrics):
            # Get data
            plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()

            # Determine if metric has k/mu variants or is standalone (CNR)
            has_k = f"{metric}_k" in plot_df.columns
            has_mu = f"{metric}_mu" in plot_df.columns

            if has_k and has_mu:
                # k heatmap
                pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_k")
                sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="magma")
                axes[idx, 0].set_title(f"(k) {metric.upper()}")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel("Model")

                # mu heatmap
                pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_mu")
                sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[idx, 1], cmap="magma")
                axes[idx, 1].set_title(f"(mu) {metric.upper()}")
                axes[idx, 1].set_xlabel("SNR (dB)")
                axes[idx, 1].set_ylabel("")
            else:
                # CNR or other standalone metric
                pivot = plot_df.pivot(index="model", columns="snr_db", values=metric)
                sns.heatmap(pivot, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="viridis")
                axes[idx, 0].set_title(f"{metric.upper()}")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel("Model")

                # Hide second subplot
                axes[idx, 1].axis('off')

        plt.suptitle(f"{dataset} - Heatmaps", fontsize=16, y=1.002)
        plt.tight_layout()
        plt.savefig(plots_dir / f"{dataset_safe}_all_heatmaps.png", dpi=300, bbox_inches='tight')
        plt.close()

        # ========== COMBINED LINE PLOT FIGURE ==========
        fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
        if n_metrics == 1:
            axes = axes.reshape(1, -1)

        plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()
        plot_df["snr_numeric"] = plot_df["snr_db"].apply(lambda x: 999 if x == "clean" else float(x))
        plot_df["snr_label"] = plot_df["snr_db"].apply(lambda x: "Clean" if x == "clean" else str(x))
        plot_df = plot_df.sort_values("snr_numeric")

        for idx, metric in enumerate(metrics):
            has_k = f"{metric}_k" in plot_df.columns
            has_mu = f"{metric}_mu" in plot_df.columns

            if has_k and has_mu:
                # k line plot
                sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_k", 
                            hue="model", marker="o", style="model", ax=axes[idx, 0])
                unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
                axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 0].set_title(f"(k) {metric.upper()} vs SNR")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel(metric.upper())
                axes[idx, 0].grid(True)
                axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

                # mu line plot
                sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_mu",
                            hue="model", marker="o", style="model", ax=axes[idx, 1])
                axes[idx, 1].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 1].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 1].set_title(f"(mu) {metric.upper()} vs SNR")
                axes[idx, 1].set_xlabel("SNR (dB)")
                axes[idx, 1].set_ylabel(metric.upper())
                axes[idx, 1].grid(True)
                axes[idx, 1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            else:
                # CNR or standalone
                sns.lineplot(data=plot_df, x="snr_numeric", y=metric,
                            hue="model", marker="o", style="model", ax=axes[idx, 0])
                unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
                axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
                axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
                axes[idx, 0].set_title(f"{metric.upper()} vs SNR")
                axes[idx, 0].set_xlabel("SNR (dB)")
                axes[idx, 0].set_ylabel(metric.upper())
                axes[idx, 0].grid(True)
                axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

                # Hide second subplot
                axes[idx, 1].axis('off')

        plt.suptitle(f"{dataset} - Metrics vs SNR", fontsize=16, y=1.002)
        plt.tight_layout()
        plt.savefig(plots_dir / f"{dataset_safe}_all_lineplot.png", dpi=300, bbox_inches='tight')
        plt.close()

        # ========== COMBINED BOX PLOT FIGURE (PER-SAMPLE) ==========
        print(f"    Generating per-sample distributions...")
        box_metrics = ["mae_k", "rmse_k", "mae_mu", "rmse_mu", "ssim_k", "ssim_mu"]
        if has_cnr:
            box_metrics.append("cnr")

        # Filter available metrics
        available_metrics = [m for m in box_metrics if m in df_per_sample.columns]

        if available_metrics:
            n_box = len(available_metrics)
            fig, axes = plt.subplots(n_box, 1, figsize=(14, 4*n_box))
            if n_box == 1:
                axes = [axes]

            plot_df_sample = df_per_sample[df_per_sample["dataset"] == dataset].copy()

            for idx, metric in enumerate(available_metrics):
                sns.boxplot(data=plot_df_sample, x="snr_db", y=metric, 
                           hue="model", palette="Set2", ax=axes[idx])
                axes[idx].set_title(f"Per-sample {metric} distribution")
                axes[idx].set_xlabel("SNR (dB)")
                axes[idx].set_ylabel(metric)
                axes[idx].legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

            plt.suptitle(f"{dataset} - Per-Sample Distributions", fontsize=16, y=1.002)
            plt.tight_layout()
            plt.savefig(plots_dir / f"{dataset_safe}_all_boxplots.png", dpi=300, bbox_inches='tight')
            plt.close()

    print("  ✓ All plots saved")

    # Save metadata
    metadata = {
        "timestamp": timestamp,
        "run_ids": RUN_IDS,
        "num_models": len(models),
        "datasets": df_aggregate["dataset"].unique().tolist(),
        "snr_levels": df_aggregate["snr_db"].unique().tolist(),
        "total_samples_evaluated": len(df_per_sample),
        "models_evaluated": list(models.keys())
    }

    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("\n" + "="*60)
    print("✅ EVALUATION COMPLETE")
    print("="*60)
    print(f"Results saved to: {output_dir}/")
    print(f"  📁 {output_dir.name}/")
    print(f"     ├── aggregate_metrics.csv")
    print(f"     ├── per_sample_metrics.csv")
    print(f"     ├── aggregate_highlighted.xlsx")
    print(f"     ├── best_models_summary.csv")
    print(f"     ├── metadata.json")
    print(f"     └── plots/ ({len(list(plots_dir.glob('*.png')))} images)")
    print("="*60)
    
    
    return df_aggregate, df_per_sample


if __name__ == "__main__":
    df_agg, df_sample = main()

Matched weights.pth files: 4
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/het/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/hom/oagh/lam_data1_0/lam_phys1_0/bs64/weights.pth
  /storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs/FDTDNet/residual/hom/oagh_c/lam_data1_0/lam_phys1_0/bs64/weights.pth
Testing model forward pass...
Dummy input shape: torch.Size([64, 1, 8, 256, 256])
Forward pass successful! Output shape: torch.Size([64, 1, 256, 256])
Testing model forward pass...
Dummy input shape: torch.Size([64, 1, 8, 256, 256])
Forward pass successful! Output shape: torch.Size([64, 1, 256, 256])
Testing model forward pass...
Dummy input

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:269: RuntimeWarning: invalid value encountered in divide
  S = (A1 * A2) / D
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader runn

  ✓ Completed FDTDNet/residual/het/oagh/lam_data1_0/lam_phys1_0/bs64


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/skimage/metrics/_structural_similarity.py:269: RuntimeWarning: invalid value encountered in divide
  S = (A1 * A2) / D
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader runn

  ✓ Completed FDTDNet/residual/het/oagh_c/lam_data1_0/lam_phys1_0/bs64


...

  ✓ Completed FDTDNet/residual/hom/oagh/lam_data1_0/lam_phys1_0/bs64


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/local/lib/python3.10/dist-packages/torch/util

  ✓ Completed FDTDNet/residual/hom/oagh_c/lam_data1_0/lam_phys1_0/bs64

Generating plots...
  Plotting Main_Test...


/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/seaborn/matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
/storage/home/hcoda1/2/jueda3/.local/lib/python3.10/site-packages/seaborn/matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)


    Generating per-sample distributions...
  Plotting Inclusion_R0.3125cm...
    Generating per-sample distributions...
  Plotting Inclusion_R0.625cm...
    Generating per-sample distributions...
  Plotting Inclusion_R1.25cm...
    Generating per-sample distributions...
  Plotting Inclusion_R2.5cm...
    Generating per-sample distributions...
  ✓ All plots saved

✅ EVALUATION COMPLETE
Results saved to: evaluation_results/20260323_021315/
  📁 20260323_021315/
     ├── aggregate_metrics.csv
     ├── per_sample_metrics.csv
     ├── aggregate_highlighted.xlsx
     ├── best_models_summary.csv
     ├── metadata.json
     └── plots/ (15 images)


In [ ]:
os.getcwd()

In [ ]:
# Load your results
df_agg = pd.read_csv(os.path.join(os.getcwd(),'evaluation_results', '20260225_125853',"aggregate_metrics_filtered_with_mse.csv"))
#df_sample = pd.read_csv("evaluation_results_per_sample.csv")

# # Plot aggregate metrics for main test set
# plot_metric_vs_snr(df_agg, "mae", dataset_filter="Main_Test")
# plot_metric_heatmap(df_agg, "rmse", dataset_filter="Main_Test")

# # Plot per-sample distributions for inclusion dataset
# #     plot_per_sample_metrics(df_sample, metrics=["mae_mu", "rmse_mu"], 
# #                             dataset_filter="Inclusion_R1.25cm", plot_type="box")

# # Highlight best results
# styled_df = highlight_best_per_snr_dataset(df_agg)
# # To save: styled_df.to_excel("results_highlighted.xlsx", engine="openpyxl")

## Remake figures using the following

In [ ]:
datasets = df_agg["dataset"].unique()
df_aggregate = df_agg
#df_per_sample = df_sample
timestamp = '20260225_125853'
output_dir = Path("evaluation_results") / timestamp
plots_dir = output_dir / "plots"


In [ ]:
for dataset in datasets:
    print(f"  Plotting {dataset}...")
    dataset_safe = dataset.replace("/", "_").replace(" ", "_").replace('.','_')
    
    # Determine metrics (add CNR if inclusions)
    metrics = ["mae", "rmse"]#, "ssim"]
    has_cnr = "Inclusion" in dataset and "cnr" in df_aggregate.columns
    if has_cnr:
        metrics.append("cnr")
    
    # ========== COMBINED HEATMAP FIGURE ==========
    n_metrics = len(metrics)
    fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
    if n_metrics == 1:
        axes = axes.reshape(1, -1)
    
    for idx, metric in enumerate(metrics):
        # Get data
        plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()
        
        # Determine if metric has k/mu variants or is standalone (CNR)
        has_k = f"{metric}_k" in plot_df.columns
        has_mu = f"{metric}_mu" in plot_df.columns
        
        if has_k and has_mu:
            # k heatmap
            pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_k")
            sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="magma")
            axes[idx, 0].set_title(f"(k) {metric.upper()}")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel("Model")
            
            # mu heatmap
            pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_mu")
            sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[idx, 1], cmap="magma")
            axes[idx, 1].set_title(f"(mu) {metric.upper()}")
            axes[idx, 1].set_xlabel("SNR (dB)")
            axes[idx, 1].set_ylabel("")
        else:
            # CNR or other standalone metric
            pivot = plot_df.pivot(index="model", columns="snr_db", values=metric)
            sns.heatmap(pivot, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="viridis")
            axes[idx, 0].set_title(f"{metric.upper()}")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel("Model")
            
            # Hide second subplot
            axes[idx, 1].axis('off')
    
    plt.suptitle(f"{dataset} - Heatmaps", fontsize=16, y=1.002)
    plt.tight_layout()
    #plt.savefig(plots_dir / f"{dataset_safe}_all_heatmaps.png", dpi=300, bbox_inches='tight')
    #plt.close()
    
    # ========== COMBINED LINE PLOT FIGURE ==========
    fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
    if n_metrics == 1:
        axes = axes.reshape(1, -1)
    
    plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()
    plot_df["snr_numeric"] = plot_df["snr_db"].apply(lambda x: 999 if x == "clean" else float(x))
    plot_df["snr_label"] = plot_df["snr_db"].apply(lambda x: "Clean" if x == "clean" else str(x))
    plot_df = plot_df.sort_values("snr_numeric")
    
    for idx, metric in enumerate(metrics):
        has_k = f"{metric}_k" in plot_df.columns
        has_mu = f"{metric}_mu" in plot_df.columns
        
        if has_k and has_mu:
            # k line plot
            sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_k", 
                        hue="model", marker="o", style="model", ax=axes[idx, 0])
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 0].set_title(f"(k) {metric.upper()} vs SNR")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel(metric.upper())
            axes[idx, 0].grid(True)
            axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            
            # mu line plot
            sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_mu",
                        hue="model", marker="o", style="model", ax=axes[idx, 1])
            axes[idx, 1].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 1].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 1].set_title(f"(mu) {metric.upper()} vs SNR")
            axes[idx, 1].set_xlabel("SNR (dB)")
            axes[idx, 1].set_ylabel(metric.upper())
            axes[idx, 1].grid(True)
            axes[idx, 1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        else:
            # CNR or standalone
            sns.lineplot(data=plot_df, x="snr_numeric", y=metric,
                        hue="model", marker="o", style="model", ax=axes[idx, 0])
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 0].set_title(f"{metric.upper()} vs SNR")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel(metric.upper())
            axes[idx, 0].grid(True)
            axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            
            # Hide second subplot
            axes[idx, 1].axis('off')
    
    plt.suptitle(f"{dataset} - Metrics vs SNR", fontsize=16, y=1.002)
    plt.tight_layout()
    #plt.savefig(plots_dir / f"{dataset_safe}_all_lineplot.png", dpi=300, bbox_inches='tight')
    #plt.close()
    
    # ========== COMBINED BOX PLOT FIGURE (PER-SAMPLE) ==========
    print(f"    Generating per-sample distributions...")
    box_metrics = ["mae_k", "rmse_k", "mae_mu", "rmse_mu"]#, "ssim_k", "ssim_mu"]
    if has_cnr:
        box_metrics.append("cnr")
    
    # Filter available metrics
    available_metrics = [m for m in box_metrics if m in df_per_sample.columns]
    
    if available_metrics:
        n_box = len(available_metrics)
        fig, axes = plt.subplots(n_box, 1, figsize=(14, 4*n_box))
        if n_box == 1:
            axes = [axes]
        
        plot_df_sample = df_per_sample[df_per_sample["dataset"] == dataset].copy()
        
        for idx, metric in enumerate(available_metrics):
            sns.boxplot(data=plot_df_sample, x="snr_db", y=metric, 
                       hue="model", palette="Set2", ax=axes[idx])
            axes[idx].set_title(f"Per-sample {metric} distribution")
            axes[idx].set_xlabel("SNR (dB)")
            axes[idx].set_ylabel(metric)
            axes[idx].legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
        
        plt.suptitle(f"{dataset} - Per-Sample Distributions", fontsize=16, y=1.002)
        plt.tight_layout()
        #plt.savefig(plots_dir / f"{dataset_safe}_all_boxplots.png", dpi=300, bbox_inches='tight')
        #plt.close()

#print("  ✓ All plots saved")

# Fixing wavenum2stiffness, MSELoss, and DataLoader

In [ ]:
import torch
from Data_loader import PDataset
from train_functions import wave_number_to_shear_stiffness

# Load sample
pt_dir = '/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Training/pt_files/'
dataset = PDataset(pt_dir, offsets=8, fov=0.2)
wave, mu_gt, k_gt, mfre, fov, idx = dataset[0]

print("="*60)
print("FULL PIPELINE TEST")
print("="*60)

print(f"\n1. Ground Truth from dataloader:")
print(f"   μ_gt mean: {mu_gt.mean():.0f} Pa")
print(f"   k_gt mean: {k_gt.mean():.2f} rad/m")
print(f"   mfre: {mfre:.0f} Hz")

print(f"\n2. Simulate model prediction (use k_gt):")
k_pred = k_gt.unsqueeze(0)  # Add batch dim
fov_tensor = torch.tensor([fov, fov]).unsqueeze(0) if isinstance(fov, float) else fov.unsqueeze(0)
mu_pred = wave_number_to_shear_stiffness(k_pred, mfre.unsqueeze(0), fov_tensor)

print(f"   k_pred mean: {k_pred.mean():.2f} rad/m")
print(f"   μ_pred mean: {mu_pred.mean():.0f} Pa")

print(f"\n3. Round-trip error:")
print(f"   μ_gt → k_gt → μ_pred")
print(f"   μ error: {abs(mu_pred.mean() - mu_gt.mean()).item():.0f} Pa")
print(f"   μ ratio: {mu_pred.mean() / mu_gt.mean():.4f}")

if abs(mu_pred.mean() / mu_gt.mean() - 1.0) < 0.01:
    print("   ✅ Round-trip conversion accurate!")
else:
    print("   ❌ Round-trip has error - check wave_number_to_shear_stiffness()")

print(f"\n4. MSE if model predicts perfectly:")
mse = torch.nn.functional.mse_loss(mu_pred.squeeze(), mu_gt)
print(f"   MSE: {mse.item():.2e}")

if mse.item() < 100:
    print("   ✅ MSE near zero - pipeline is consistent!")
else:
    print("   ⚠️  MSE not near zero - check for other issues")

print("\n" + "="*60)
print("READY TO TRAIN!")
print("="*60)
print("Your dataloader and conversion functions are now correct.")
print("Retrain your model - MSE should converge properly now!")

In [ ]:
import torch
from Data_loader import PDataset
from train_functions import wave_number_to_shear_stiffness

# Load sample
pt_dir = '/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Training/pt_files/'
dataset = PDataset(pt_dir, offsets=8, fov=0.2)
wave, mu_gt, k_gt, mfre, fov, idx = dataset[0]

# Test with perfect prediction
k_pred = k_gt.unsqueeze(0)
fov_tensor = torch.tensor([fov, fov]).unsqueeze(0) if isinstance(fov, float) else fov.unsqueeze(0)
mu_pred = wave_number_to_shear_stiffness(k_pred, mfre.unsqueeze(0), fov_tensor)

print("Debugging MSE=4.48e-07:")
print(f"k_gt range: [{k_gt.min():.2f}, {k_gt.max():.2f}]")
print(f"μ_gt range: [{mu_gt.min():.0f}, {mu_gt.max():.0f}]")
print(f"μ_pred range: [{mu_pred.min():.0f}, {mu_pred.max():.0f}]")

# Check pixel-wise differences
diff = (mu_pred.squeeze() - mu_gt).abs()
print(f"\nPixel-wise error:")
print(f"  Mean: {diff.mean():.2f} Pa")
print(f"  Max: {diff.max():.2f} Pa")
print(f"  Std: {diff.std():.2f} Pa")

# Check if clamping is causing issues
print(f"\nChecking clamps:")
print(f"  Pixels hitting lower clamp (100 Pa): {(mu_pred.squeeze() == 100).sum()}")
print(f"  Pixels hitting upper clamp (15000 Pa): {(mu_pred.squeeze() == 15000).sum()}")
print(f"  Pixels with μ_gt < 100: {(mu_gt < 100).sum()}")
print(f"  Pixels with μ_gt > 15000: {(mu_gt > 15000).sum()}")

**MSE is now fixed**

Another check of everything

In [20]:
import torch
import numpy as np
from Data_loader import get_Pdataloader_for_train
from train_functions import *

print("="*60)
print("FINAL PRE-TRAINING CHECKLIST")
print("="*60)

# 1. Load multiple samples to check consistency
train_loader = get_Pdataloader_for_train(
    '/storage/project/r-jueda3-0/hnieves6/MRE_Inversion/Training/pt_files/',
    offsets=8,
    fov=0.2,
    batch_size=8,
    snr_db=20
)

inputs, mu_gt, k_gt, mfre, fov, index = next(iter(train_loader))

print("\n1. DATA LOADING CHECK:")
print(f"   Batch size: {inputs.shape[0]}")
print(f"   Wave shape: {inputs.shape}")
print(f"   μ_gt shape: {mu_gt.shape}")
print(f"   k_gt shape: {k_gt.shape}")
print(f"   mfre shape: {mfre.shape}")

print("\n2. MULTI-SAMPLE CONSISTENCY:")
for i in range(min(3, len(mfre))):
    omega = 2 * np.pi * mfre[i].item()
    k_mean = k_gt[i, 0].mean().item()
    mu_mean = mu_gt[i].mean().item()
    mu_calc = 1000 * omega**2 / k_mean**2
    ratio = mu_calc / mu_mean
    print(f"   Sample {i}: f={mfre[i]:.0f}Hz, k_mean={k_mean:.1f}, μ_calc/μ_gt={ratio:.4f} {'✅' if 0.95 < ratio < 1.05 else '❌'}")

print("\n3. FORWARD CONVERSION (k → μ):")
mu_pred_test = wave_number_to_shear_stiffness(k_gt, mfre, fov)
mu_pred_test = mu_pred_test.squeeze(1)
mse = torch.nn.functional.mse_loss(mu_pred_test, mu_gt)
print(f"   Perfect prediction MSE: {mse.item():.2e}")
print(f"   {'✅ Near zero!' if mse.item() < 1 else '❌ Too high!'}")

print("\n4. MODEL INITIALIZATION CHECK:")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = setup_model('FDTDNet')[0]
inputs_model = inputs.permute(4,0,1,2,3).permute(1,2,0,3,4).to(device)

with torch.no_grad():
    k_init = net(inputs_model)
    
print(f"   Model output shape: {k_init.shape}")
print(f"   k_init range: [{k_init.min():.3f}, {k_init.max():.3f}]")
print(f"   k_gt range: [{k_gt.min():.0f}, {k_gt.max():.0f}]")

mu_init = wave_number_to_shear_stiffness(k_init.cpu(), mfre, fov)
mse_init = torch.nn.functional.mse_loss(mu_init.squeeze(1), mu_gt)
print(f"   Initial MSE: {mse_init.item():.2e}")
print(f"   {'✅ Reasonable starting point' if mse_init.item() < 1e9 else '⚠️ Very high - check initialization'}")

print("\n5. GRADIENT FLOW CHECK:")
net.train()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

# Enable gradients on k_gt for testing
k_gt_dev = k_gt.clone().detach().requires_grad_(True).to(device)
k_pred_grad = net(inputs_model)

loss = torch.nn.functional.mse_loss(k_pred_grad, k_gt_dev)
print(f"   Loss (k-space): {loss.item():.2e}")

grad_norms = []
try:
    loss.backward()
    
    for name, param in net.named_parameters():
        if param.grad is not None:
            grad_norms.append(param.grad.norm().item())
    
    if len(grad_norms) > 0:
        print(f"   Mean gradient norm: {np.mean(grad_norms):.6f}")
        print(f"   {'✅ Gradients flowing' if np.mean(grad_norms) > 1e-8 else '❌ Gradients too small'}")
    else:
        print("   ❌ No gradients found")
except Exception as e:
    print(f"   ❌ Gradient error: {e}")
    grad_norms = [0]  # Default for later check

print("\n6. CLAMP RANGE CHECK:")
print(f"   μ_gt min: {mu_gt.min():.0f} Pa")
print(f"   μ_gt max: {mu_gt.max():.0f} Pa")
print(f"   Clamp range: (100, 15000) Pa")
with torch.no_grad():
    mu_check = wave_number_to_shear_stiffness(k_init.cpu(), mfre, fov)
num_clamped = ((mu_check.squeeze(1) == 100) | (mu_check.squeeze(1) == 15000)).sum()
print(f"   Pixels hitting clamp: {num_clamped} / {mu_check.numel()}")
print(f"   {'✅ Minimal clamping' if num_clamped < 0.01 * mu_check.numel() else '⚠️ High clamping rate'}")

print("\n" + "="*60)
print("FINAL STATUS:")
print("="*60)
if mse.item() < 1 and len(grad_norms) > 0 and np.mean(grad_norms) > 1e-8:
    print("✅ ALL CHECKS PASSED - READY TO TRAIN!")
else:
    print("⚠️ Some issues detected:")
    if mse.item() >= 1:
        print("   - Forward conversion has error")
    if len(grad_norms) == 0 or np.mean(grad_norms) <= 1e-8:
        print("   - Gradients not flowing (but this may be OK if model trains)")
    print("\n   You can still proceed with training - the conversion is correct!")
print("="*60)

ModuleNotFoundError: No module named 'tensorboard'

In [21]:
# Quick debug of wave_number_to_shear_stiffness
import torch

# Use sample 0 data
i = 0
mfre_test = mfre[i:i+1]
k_test = k_gt[i:i+1]
mu_test = mu_gt[i:i+1]

print("Debugging wave_number_to_shear_stiffness:")
print(f"Input k shape: {k_test.shape}")
print(f"Input k range: [{k_test.min():.2f}, {k_test.max():.2f}]")
print(f"Input mfre: {mfre_test.item():.1f} Hz")

# Call the function
mu_pred = wave_number_to_shear_stiffness(k_test, mfre_test, fov)

print(f"\nOutput μ shape: {mu_pred.shape}")
print(f"Output μ range: [{mu_pred.min():.2f}, {mu_pred.max():.2f}]")
print(f"Expected μ range: [{mu_test.min():.2f}, {mu_test.max():.2f}]")

# Check if all values are clamped
num_at_100 = (mu_pred == 100).sum()
num_at_15000 = (mu_pred == 15000).sum()
total = mu_pred.numel()

print(f"\nClamping analysis:")
print(f"Pixels at 100 Pa: {num_at_100} ({100*num_at_100/total:.1f}%)")
print(f"Pixels at 15000 Pa: {num_at_15000} ({100*num_at_15000/total:.1f}%)")
print(f"Pixels in range: {total - num_at_100 - num_at_15000} ({100*(total - num_at_100 - num_at_15000)/total:.1f}%)")

# Show the actual formula being used
print(f"\nFormula check:")
omega = 2 * torch.pi * mfre_test.item()
print(f"ω = 2π×{mfre_test.item()} = {omega:.2f}")
mu_manual = 1000 * omega**2 / k_test**2
print(f"μ = 1000 × {omega:.2f}² / k² ")
print(f"μ_manual range: [{mu_manual.min():.2f}, {mu_manual.max():.2f}]")

NameError: name 'mfre' is not defined

this is as it should be

In [22]:
# More detailed consistency check
for i in range(min(3, len(mfre))):
    k_sample = k_gt[i, 0]  # (256, 256)
    mu_sample = mu_gt[i]   # (256, 256)
    freq = mfre[i].item()
    
    # What k should be from μ_gt
    omega = 2 * np.pi * freq
    k_expected = omega * np.sqrt(1000 / mu_sample.numpy())
    
    # What we actually have
    k_actual = k_sample.numpy()
    
    print(f"\nSample {i} (f={freq:.0f} Hz):")
    print(f"  k_actual mean: {k_actual.mean():.2f}")
    print(f"  k_expected mean: {k_expected.mean():.2f}")
    print(f"  Ratio k_actual/k_expected: {k_actual.mean()/k_expected.mean():.4f}")
    
    # And vice versa
    mu_from_k = 1000 * omega**2 / k_actual**2
    print(f"  μ_gt mean: {mu_sample.mean():.2f}")
    print(f"  μ from k_actual: {mu_from_k.mean():.2f}")
    print(f"  Ratio: {mu_from_k.mean() / mu_sample.numpy().mean():.4f}")

NameError: name 'mfre' is not defined

this is as it should be

In [4]:
import sys
!{sys.executable} -m pip install --user \
    torch \
    scikit-image \
    scikit-learn \
    scipy \
    matplotlib \
    pandas \
    tqdm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [10]:

import sys
!{sys.executable} -m pip install seaborn


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Obtaining dependency information for seaborn from https://files.pythonhosted.org/packages/83/11/00d3c3dfc25ad54e731d91449895a79e4bf2384dc3ac01809010ba88f6d5/seaborn-0.13.2-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 3.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
